# 命名实体识别 NER：从文本中提取实体
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：命名实体识别.py | 核心功能：序列标注、BIO 标注、CRF/HMM

## 概述

NER 从文本中识别出人名、地名、组织名等实体。本质是序列标注问题——给每个 token 打标签（B-PER, I-PER, O 等）。

## 数学原理

### 1. NER 的序列标注框架

命名实体识别（NER）本质上是**序列标注**问题：给定输入序列 $\mathbf{x} = (x_1, x_2, \ldots, x_n)$，预测标签序列 $\mathbf{y} = (y_1, y_2, \ldots, y_n)$。

标签使用 **BIO 标注方案**：
- `B-XXX`：实体 XXX 的开始
- `I-XXX`：实体 XXX 的内部
- `O`：非实体

### 2. 基于规则的 NER

使用正则表达式匹配特定模式：

$$\text{pattern} = \text{regex}(\text{"[一-龥]{2,6}大学"})$$

匹配结果：`P(\text{"北京大学"} | \text{pattern}) = 1`

局限性：
- 需要人工编写规则
- 覆盖率有限
- 难以处理歧义

### 3. 基于 CRF 的 NER

条件随机场（CRF）建模标签序列的联合概率：

$$P(\mathbf{y}|\mathbf{x}) = \frac{1}{Z(\mathbf{x})} \exp\left(\sum_{i=1}^{n}\sum_k \lambda_k f_k(y_{i-1}, y_i, \mathbf{x}, i)\right)$$

其中 $f_k$ 是特征函数，$\lambda_k$ 是权重，$Z(\mathbf{x})$ 是配分函数。

CRF 考虑标签之间的转移约束（如 `I-PER` 不能跟在 `B-LOC` 后面）。

### 4. 基于 BERT 的 NER

BERT 使用 Transformer 编码器，为每个 token 生成上下文表示：

$$\mathbf{h}_i = \text{BERT}(\mathbf{x})_i \in \mathbb{R}^{768}$$

然后对每个 token 的表示做分类：

$$P(y_i | \mathbf{x}) = \text{softmax}(W \mathbf{h}_i + b)$$

### 5. BIO 标注的数学约束

合法的标签序列满足：

$$P(y_i = \text{I-XXX} | y_{i-1} = \text{B-YYY}) = 0, \quad \text{当 XXX} \neq \text{YYY}$$

$$P(y_i = \text{I-XXX} | y_{i-1} = \text{O}) = 0$$

CRF 的转移矩阵显式建模这些约束。

### 6. 评估指标

NER 的评估通常基于**实体级别**的精确匹配：

- **精确率（Precision）**：预测实体中正确的比例
- **召回率（Recall）**：真实实体中被找到的比例
- **F1 值**：精确率和召回率的调和平均

$$F1 = \frac{2 \cdot P \cdot R}{P + R}$$

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `re.compile(pattern)` | 正则表达式规则匹配 |
| `AutoTokenizer.from_pretrained("bert-base-chinese")` | BERT 分词器 |
| `tokenizer(text, return_tensors="pt")` | 文本 $\to$ token ID 序列 |
| `convert_ids_to_tokens()` | ID $\to$ 可读 token |
| `("张", "B-PER")` | BIO 标注：$y_i = \text{B-PER}$ |
| `("京", "I-LOC")` | 实体内部：$y_i = \text{I-LOC}$ |

### 1. 基于规则的 NER

运行 1. 基于规则的 NER 的代码，观察算法在该环节的行为。

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import re

print("=== 基于规则的 NER ===")
text = "张三在北京大学工作,他毕业于清华大学,住在北京市海淀区。"
print(f"文本: {text}")

# 简单规则：匹配"XX大学"、"XX市"
patterns = {
    "组织": re.compile(r"[一-龥]{2,6}(?:大学|公司|研究院|实验室)"),
    "地点": re.compile(r"[一-龥]{2,6}(?:市|省|区|县|镇)"),
}
for entity_type, pattern in patterns.items():
# --- 继续 ---
    entities = pattern.findall(text)
    print(f"  {entity_type}: {entities}")

### 2. 基于预训练模型的 NER

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    import torch
    HAS_TF = True
except ImportError:
# --- 赋值/配置 ---
    HAS_TF = False
    print("[SKIP] transformers 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_TF = False
    print("\ntransformers 未安装,跳过预训练模型 NER 演示")

if HAS_TF:
    # 使用中文 BERT NER 模型
    model_name = "bert-base-chinese"
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        # 使用简单的标注示例代替专门的 NER 模型
        print("\n=== BERT 分词示例 ===")
        text_en = "Apple Inc. is located in California."
        inputs = tokenizer(text_en, return_tensors="pt")
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        print(f"  原文: {text_en}")
# --- 输出结果 ---
        print(f"  Token: {tokens}")
    except Exception as e:
        print(f"  模型加载失败: {e}")

### 3. CoNLL 格式标注数据

运行 3. CoNLL 格式标注数据 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== CoNLL 标注格式 (BIO) ===")
# B-: 实体开始, I-: 实体内部, O: 非实体
conll_data = [
    ("张", "B-PER"), ("三", "I-PER"), ("在", "O"), ("北", "B-LOC"),
    ("京", "I-LOC"), ("大", "I-LOC"), ("学", "I-LOC"), ("工", "O"),
    ("作", "O"),
]
# --- 循环处理 ---
for word, tag in conll_data:
    print(f"  {word}\t{tag}")

print("\n常见标签:")
print("  B-PER / I-PER: 人名")
print("  B-LOC / I-LOC: 地名")
print("  B-ORG / I-ORG: 组织名")
print("  B-TIME / I-TIME: 时间")
# --- 输出结果 ---
print("  O: 非实体")

### 4. 简化 NER 模型（BiLSTM-CRF）

运行 4. 简化 NER 模型（BiLSTM-CRF） 的代码，观察算法在该环节的行为。

In [ ]:
import torch
import torch.nn as nn

class SimpleNERModel(nn.Module):
    """简化版 BiLSTM NER 模型"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
# --- 继续 ---
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.hidden2tag = nn.Linear(hidden_dim * 2, n_tags)

    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds)
        logits = self.hidden2tag(lstm_out)
        return logits

print("\n=== BiLSTM NER 模型 ===")
model = SimpleNERModel(vocab_size=5000, embed_dim=128, hidden_dim=64, n_tags=7)
total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,}")
print(f"结构: Embedding → BiLSTM → Linear → CRF(可选)")

### 5. NER 评估指标

在测试集上评估模型性能，关注关键指标。

In [ ]:
print("\n=== NER 评估指标 ===")
print("实体级别评估（严格匹配）:")
print("  精确率: 预测实体中正确的比例")
print("  召回率: 真实实体中被找到的比例")
print("  F1: 精确率和召回率的调和平均")
# --- 输出结果 ---
print()
print("片段级别评估:")
print("  只要实体类型正确就算对（不要求边界完全匹配）")

### 6. 常用 NER 工具和模型

运行 6. 常用 NER 工具和模型 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 常用 NER 工具 ===")
print("1. spaCy: 工业级 NLP 工具,内置 NER")
print("   en_core_web_sm / zh_core_web_sm")
print("2. Hugging Face Transformers:")
print("   bert-base-chinese-finetuned-cluener")
# --- 输出结果 ---
print("3. HanLP: 中文 NLP 工具包,NER 效果好")
print("4. LAC (百度): 中文词法分析（分词+词性+NER）")

print("\n=== NER 要点 ===")
print("- 中文 NER 需要先分词（分词错误会传播）")
print("- BERT + CRF 是当前最强的 NER 架构之一")
print("- 标注数据是 NER 的主要瓶颈（标注成本高）")
print("- 少样本场景: 尝试 few-shot NER 或数据增强")
# --- 输出结果 ---
print("- 实体嵌套问题: 传统 BIO 标注难以处理嵌套实体")

## 关键代码解释

```python
# BIO 标注示例
# "张三去北京" → [B-PER, I-PER, O, B-LOC, I-LOC]
```

## 注意事项

1. 嵌套实体是难点（"北京大学" 既是组织又是地点）
2. 标注一致性很重要
3. 领域迁移困难

## 总结与延伸

以上代码展示了 **命名实体识别** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **BERT + CRF**：当前 NER 的主流架构
- **少样本 NER**：用大模型做 few-shot NER
- **关系抽取**：NER 的下游任务
